In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-generative" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


## Grid Graph Generator (Mexican Hat)
Generate a labeled 2D grid graph from a 2D function, then visualize both the continuous values and discretized labels.


In [ ]:
# Grid utilities: 2D function evaluation and discretized labeled grids
import numpy as np
import networkx as nx

def mexican_hat_2d(x: np.ndarray, y: np.ndarray, *, sigma: float = 1.0) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    r2 = x * x + y * y
    s2 = sigma * sigma
    return (1.0 - r2 / s2) * np.exp(-0.5 * r2 / s2)


def _linspace_grid(width: int, height: int, x_range, y_range):
    xs = np.linspace(float(x_range[0]), float(x_range[1]), num=max(1, int(width)))
    ys = np.linspace(float(y_range[0]), float(y_range[1]), num=max(1, int(height)))
    X, Y = np.meshgrid(xs, ys)
    return X, Y


def _discretize_values(values: np.ndarray, *, n_bins: int = 3, method: str = 'quantile', clip=None):
    if n_bins < 2:
        raise ValueError('n_bins must be >= 2')
    v = np.asarray(values, dtype=float)
    if clip is not None:
        v = np.clip(v, clip[0], clip[1])
    flat = v.reshape(-1)
    if method == 'quantile':
        quantiles = np.linspace(0.0, 1.0, num=n_bins + 1)
        edges = np.quantile(flat, quantiles)
        if np.any(np.diff(edges) <= 0):
            vmin, vmax = float(np.min(flat)), float(np.max(flat))
            edges = np.linspace(vmin, vmax, num=n_bins + 1)
    elif method == 'uniform':
        vmin, vmax = float(np.min(flat)), float(np.max(flat))
        edges = np.linspace(vmin, vmax, num=n_bins + 1)
    else:
        raise ValueError("method must be 'quantile' or 'uniform'")
    labels = np.digitize(flat, edges[1:-1], right=False)
    labels = np.clip(labels, 0, n_bins - 1).astype(int)
    labels = labels.reshape(v.shape)
    return labels, edges


def generate_labeled_grid_graph(
    width: int,
    height: int,
    *,
    func= mexican_hat_2d,
    x_range=(-2.0, 2.0),
    y_range=(-2.0, 2.0),
    n_bins: int = 3,
    discretize_method: str = 'quantile',
    add_diagonals: bool = False,
    store_position: bool = True,
):
    if width <= 0 or height <= 0:
        raise ValueError('width and height must be positive')
    G = nx.grid_2d_graph(height, width)
    if add_diagonals:
        for i in range(height - 1):
            for j in range(width - 1):
                G.add_edge((i, j), (i + 1, j + 1))
                G.add_edge((i + 1, j), (i, j + 1))
    X, Y = _linspace_grid(width=width, height=height, x_range=x_range, y_range=y_range)
    V = func(X, Y)
    if V.shape != X.shape:
        raise ValueError('func must return an array with shape (height, width)')
    labels, edges = _discretize_values(V, n_bins=n_bins, method=discretize_method)
    for i in range(height):
        for j in range(width):
            node = (i, j)
            attrs = {'value': float(V[i, j]), 'label': int(labels[i, j])}
            if store_position:
                attrs['pos'] = (float(X[i, j]), float(Y[i, j]))
            G.nodes[node].update(attrs)
    # Assign edge 'label' as concatenation of endpoint labels (sorted)
    for u, v in G.edges():
        lu = G.nodes[u].get('label')
        lv = G.nodes[v].get('label')
        if lu is None or lv is None:
            continue
        a, b = (lu, lv) if lu <= lv else (lv, lu)
        G.edges[u, v]['label'] = '-' #f'{a}:{b}'
    G.graph['value_edges'] = edges.astype(float)
    G.graph['x_range'] = tuple(map(float, x_range))
    G.graph['y_range'] = tuple(map(float, y_range))
    G.graph['n_bins'] = int(n_bins)
    G.graph['discretize_method'] = str(discretize_method)
    return G


def generate_mexican_hat_grid(
    width: int,
    height: int,
    *,
    sigma: float = 1.0,
    x_range=(-2.0, 2.0),
    y_range=(-2.0, 2.0),
    n_bins: int = 3,
    discretize_method: str = 'quantile',
    add_diagonals: bool = False,
    store_position: bool = True,
):
    def _f(xx, yy):
        return mexican_hat_2d(xx, yy, sigma=sigma)
    return generate_labeled_grid_graph(
        width=width,
        height=height,
        func=_f,
        x_range=x_range,
        y_range=y_range,
        n_bins=n_bins,
        discretize_method=discretize_method,
        add_diagonals=add_diagonals,
        store_position=store_position,
    )

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Parameters
width, height = 8,8
sigma = .5
n_bins = 7
x_range = (-2.0, 2.0)
y_range = (-2.0, 2.0)

G = generate_mexican_hat_grid(
    width, height,
    sigma=sigma,
    n_bins=n_bins,
    x_range=x_range,
    y_range=y_range,
    discretize_method='quantile',
    add_diagonals=False,
    store_position=True,
)

pos = nx.get_node_attributes(G, 'pos')

# Extract arrays for heatmaps
vals = np.zeros((height, width), dtype=float)
labs = np.zeros((height, width), dtype=int)
for i in range(height):
    for j in range(width):
        d = G.nodes[(i, j)]
        vals[i, j] = d['value']
        labs[i, j] = d['label']

vals.shape, labs.shape

In [ ]:
# Heatmaps: continuous values and discretized labels
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(
    vals, cmap='viridis', origin='lower',
    extent=[x_range[0], x_range[1], y_range[0], y_range[1]]
)
axes[0].set_title('Mexican hat value')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(
    labs, cmap='tab10', origin='lower', vmin=0, vmax=n_bins-1,
    extent=[x_range[0], x_range[1], y_range[0], y_range[1]]
)
axes[1].set_title(f'Discretized labels ({n_bins} bins)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
plt.tight_layout()

In [ ]:
# Draw the grid graph colored by label (using stored positions)
from abstractgraph.display import display_graph
G = nx.convert_node_labels_to_integers(G)
_ = display_graph(
    G,
    style={'node_size': 40, 'edge_width': 1, 'node_alpha': 0.9, 'edge_color': 'k', 'cmap': 'tab10'},
)

## Random Vertex Removals
Create multiple copies of `G` by removing a fixed number of randomly-sampled vertices.


In [ ]:
# Perturbation utility: random vertex removals
import networkx as nx
import random as _random

def _is_connected_any(G: nx.Graph) -> bool:
    if G.number_of_nodes() == 0:
        return True
    if nx.is_directed(G):
        try:
            return nx.is_weakly_connected(G)
        except Exception:
            return True
    else:
        try:
            return nx.is_connected(G)
        except Exception:
            return True

def random_vertex_removal_copies(
    G: nx.Graph,
    *,
    n_remove: int,
    n_samples: int,
    seed: int | None = None,
    connected_only: bool = False,
    max_attempts_per_sample: int = 100,
    return_removed: bool = False,
):
    if n_samples < 1:
        raise ValueError('n_samples must be >= 1')
    n = G.number_of_nodes()
    if n <= 0:
        raise ValueError('Input graph has no nodes.')
    if not (1 <= n_remove < n):
        raise ValueError('n_remove must satisfy 1 <= n_remove < number of nodes in G')
    rng = _random.Random(seed)
    nodes = list(G.nodes())
    out_graphs = []
    removed_list = []
    for _ in range(n_samples):
        attempts = 0
        while True:
            attempts += 1
            removed = tuple(rng.sample(nodes, k=n_remove))
            H = G.copy()
            H.remove_nodes_from(removed)
            if not connected_only or _is_connected_any(H):
                out_graphs.append(H)
                removed_list.append(removed)
                break
            if attempts >= max_attempts_per_sample:
                out_graphs.append(H)
                removed_list.append(removed)
                break
    return (out_graphs, removed_list) if return_removed else out_graphs

In [ ]:

n_remove = 2   # number of nodes to remove per sample
n_samples = 1000    # how many perturbed copies to generate

graphs = random_vertex_removal_copies(
    G,
    n_remove=n_remove,
    n_samples=n_samples,
    seed=123,
    connected_only=True,
    return_removed=False,
)
from abstractgraph.hashing import GraphHashDeduper
graphs = GraphHashDeduper().fit_filter(graphs)
print(f'#graphs:{len(graphs)}')

In [ ]:
# Visualize one perturbed example using the original positions
from abstractgraph.display import display_graphs
import networkx as nx
import random

n_graphs_per_line = 7
H = random.sample(graphs, k=min(n_graphs_per_line*3, len(graphs)))
_ = display_graphs(
    H,
    style={'node_size': 40, 'edge_width': 1, 'node_alpha': 0.9, 'edge_color': 'k', 'cmap': 'tab10'},
    n_graphs_per_line=n_graphs_per_line
)

In [ ]:

from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator, FeasibilityEstimatorNumberOfNodesInRange
from abstractgraph_generative.interpolate import InterpolationEstimator
from abstractgraph_generative.interpolation import InterpolationGenerator
from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import RandomForestClassifier
from abstractgraph.vectorize import AbstractGraphTransformer

nbits=14

decomposition_function = neighborhood(radius=(1,3))
graph_transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=decomposition_function,
    return_dense=True,
)

#df = compose(filter_by_number_of_nodes(number_of_nodes=4), cycle())
df = compose(add(neighborhood(radius=2), cycle()), unlabel())

fe = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
feasibility_estimators = [fe]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)
feasibility_estimator.fit(graphs)

interpolation_estimator = InterpolationEstimator(
    graph_transformer=graph_transformer,
    n_iterations=3,
    n_samples=100,
    k=5,
    feasibility_estimator=feasibility_estimator,
    degree_penalty="auto",
    degree_penalty_mode="multiplicative",
    cut_radius=1,
    cut_scope="both",
)
interpolation_estimator.fit(graphs)

In [ ]:
from abstractgraph_generative.rewrite import rewrite
source = graphs[1]
donors = graphs[3:4]
samples = rewrite(source=source, donors=donors, decomposition_function=decomposition_function, nbits=nbits, n_samples=1, single_replacement=True, cut_radius=1, cut_scope="both", cut_include_edge_label=True)
print(f'#samples:{len(samples)}')
samples = GraphHashDeduper().fit_filter(samples)
samples = GraphHashDeduper().fit([source]+donors).filter(samples)
print(f'#unique samples:{len(samples)}')
_ = display_graphs([source]+donors)
_ = display_graphs(samples[:7])
filtered_samples = feasibility_estimator.filter(samples)
print(f'#filtered:{len(filtered_samples)}')
_ = display_graphs(filtered_samples[:7])

In [ ]:
import random
source_idx=random.randrange(len(graphs))
destination_idx=random.randrange(len(graphs))
source,destination = graphs[source_idx], graphs[destination_idx]
_ = display_graphs([source,destination])
samples = interpolation_estimator.interpolate_idxs(source_idx, destination_idx)
print(f'#generated:{len(samples)}')
samples = GraphHashDeduper().fit_filter(samples)
samples = GraphHashDeduper().fit([source, destination]).filter(samples)
print(f'#accepted:{len(samples)}')
if samples:
    _ = display_graphs([source] + samples + [destination])
else:
    print('No novel results')

---

In [ ]:
generator = InterpolationGenerator(interpolation_estimator).fit(graphs)

from abstractgraph.display import display_graphs
generated_graphs, generated_targets = generator.generate(
    n_rounds=4,
    n_pairs=5,
    n_iterations=3,
    best_of=1,
    prob_for_acceptance_interval=(0.51, 0.95),
    verbose=True,
    draw_func=display_graphs,
)